In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

print(np.__version__)

2.3.5


In [ ]:
#Load Data
CSV_DATA = 'food_prices.csv'

#adding filtered data
RECENT_DATA = 'food_price_2020_2026.csv'

new_df = pd.read_csv(RECENT_DATA)

df_raw = pd.read_csv(CSV_DATA)
print(f'totla records : {len(df_raw)}')
print(f'columns : {list(df_raw.columns)}' )
print(f'Date range : {df_raw["date"].min()} -> {df_raw["date"].max()}')

print(f'\nfirst 4 rows')

df_raw.head(4)

print(f'\nNEW_DATA: \n{new_df.head(5)}')


In [ ]:
#Check geographical coverage

print(f'\nRegions (admin1:)')
print(df_raw["admin1"].value_counts(dropna=False))

print(f'\nDistricts (admin2:)')
district = df_raw["admin2"].unique()
print(district)
print(f'\nDistricts: (admin2) - {df_raw["admin2"].nunique()} unique')

print(f'\n {sorted(df_raw["admin2"].dropna().unique())}')
len(df_raw["admin2"].dropna().unique())


print(f'\nMarket : - {df_raw["market"].nunique()} unique')
# df_raw["market"].nunique()

print("\n")
print(f'\nRecords with coordinates: {df_raw[["latitude", "longitude"]].notna().all(axis=1).sum()}')
print(f'\nRecords without coordinates either x or y: {df_raw[["latitude", "longitude"]].isna().any(axis=1).sum()} ')
print(f'\nRecords without coordinates both x and y: {df_raw[["latitude","longitude"]].isna().all(axis=1).sum()}')


#____________CHECK COMMODITY COVERAGE____________
print(df_raw.groupby("category")["commodity"].nunique().to_string())
print('\n')
print(df_raw.groupby(["category", "commodity"]).size().reset_index(name='records').to_string(index=False))

In [ ]:
#___________DATA CLEANING____________________

#filter data only from 2020 - 2026 focus and category 

df = df_raw.copy()

#parse dates propery
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year

#Keep only food category drop(non-food, fuels and exchange rate)
FOOD_CATEGORIES = [
    'cereals and tubers',
    'pulses and nuts',
    'meat, fish and eggs',
    'miscellaneous food',
    'oil and fats',
    'vegetables and fruits'
]

# df = df[df["category"].isin(FOOD_CATEGORIES)]
# print(f'after food category filter: {len(df)}')

#Filter data from 2020 -2026 and only food categorie

df_recent = df[
    (df["date"] >= '2020-01-01') &
    (df["date"] <= '2026-04-15') &
    (df["category"].isin(FOOD_CATEGORIES))
]
len(df_recent)
df_recent.to_csv("food_price_2020_2026.csv", index=False)

print(f'\nNew data frame filtered: {len(new_df)}')

df = new_df.copy()
len(df)

# Drop the "National Average" aggregated rows — we want market-level data

# only aggregated national average data

#    (We keep national average separately for benchmarking later)
df_national = df[df["market"] == 'National Average'].copy()
print(f'\n national average records kept separtetly: {len(df_national)} records')

df = df[df["market"] != "National Average"]
print(f'\nAfter removing national average: {len(df):,} records')

# 4. Drop rows with no district name (we need admin2 for spatial joins)
# df = df.dropna(subset=["admin2"])
print(f'After dropping no-district rows: {len(df):,} records')

# Drop rows with no price
df = df.dropna(subset=["price"])
print(f'After dropping no-price rows: {len(df):,} records')

# Standardise column names
df = df.rename(columns={
    'admin1': 'region',
    'admin2': 'district'
})


#print(df.columns.tolist())

# Sort by location + commodity + date — essential for time-series diff
df = df.sort_values(['district', 'market', 'commodity', 'date']).reset_index(drop=True)

print('\n Cleaning complete')
df[['date','region','district','market','commodity','category','price','latitude','longitude']].head(5)

print(df['unit'].value_counts())

print(df.groupby(['commodity', 'unit']).size().reset_index(name='count').to_string(index=False))

# we have the problem in units which might cause spike problems eg fish in malawi context are sold in heap no KG
# Fully standardised — safe for spike detection and price comparison
# first lets ovewrite the csv with new data 

df.to_csv("food_price_2020_2026.csv", index=False)

In [ ]:
#______FOCUS  UNIT COLUMN DATA CLEANING___

print(df["unit"].value_counts())
print(f'\n')
df.groupby(["commodity","unit"]).size().reset_index(name="count")

# See every commodity + unit combination we need to fix
print(df.groupby(['commodity', 'unit'])['price'].agg(['count', 'mean', 'min', 'max']).round(0).to_string())

In [ ]:
# ── Approach: Two clean files ───────────────────────────────────────────────

# File 1: Standardised — KG and Litres only
# These are your spike detection and choropleth data
df_standard = df[df['unit'].isin(['KG', 'L'])].copy()

# Fix the one real problem — Groundnuts outlier
outlier_mask = (
    (df_standard['commodity'] == 'Groundnuts (shelled)') &
    (df_standard['price'] > 25000)
)
print(f'Groundnuts outliers removed: {outlier_mask.sum()} rows')
df_standard = df_standard[~outlier_mask]

# File 2: Informal — Heap, Bunch, Unit kept exactly as collected
df_informal = df[df['unit'].isin(['Heap', 'Bunch', 'Unit'])].copy()

# Add price index per commodity (% change from first observation)
# This lets you track trends WITHOUT converting units
df_informal = df_informal.sort_values(
    ['district', 'market', 'commodity', 'date']
).reset_index(drop=True)

df_informal['price_pct_change'] = (
    df_informal
    .groupby(['district', 'market', 'commodity'])['price']
    .pct_change() * 100
)

# ── Summary ───────────────────────────────────────────────────────────────────
print('=== FILE 1: Standardised (KG + L) ===')
print(f'Records     : {len(df_standard):,}')
print(f'Commodities : {df_standard["commodity"].nunique()}')
print(sorted(df_standard["commodity"].unique()))

print('\n=== FILE 2: Informal (Heap + Bunch + Unit) ===')
print(f'Records     : {len(df_informal):,}')
print(f'Commodities : {df_informal["commodity"].nunique()}')
print(sorted(df_informal["commodity"].unique()))

# ── Save ──────────────────────────────────────────────────────────────────────
df_standard.to_csv('food_prices_standardised.csv', index=False)
df_informal.to_csv('food_prices_informal.csv', index=False)

print('\nfood_prices_standardised.csv → use for spike detection')
print('food_prices_informal.csv     → use for informal market layer')